# Word2Vec – Przykłady użycia modelu

Notebook zakłada, że masz już wytrenowany model z `word2vec_tutorial.ipynb`.  
Jeśli nie – uruchom najpierw tamten notebook i wróć tutaj.

In [ ]:
from gensim.models import Word2Vec
import numpy as np

model = Word2Vec.load("wolne_lektury_w2v.model")
wv = model.wv

print(f"Model załadowany: {len(wv):,} słów, wektory {wv.vector_size}D")

---
## 1. Eksploracja słownika

In [ ]:
# Najczęstsze słowa w słowniku
# wv.index_to_key – lista słów posortowana od najczęstszych
print("Top 30 najczęstszych słów:")
print(wv.index_to_key[:30])

In [ ]:
# Sprawdzanie czy słowo jest w słowniku
slowa = ["miłość", "python", "król", "smartphone", "rycerz", "komputer"]

for s in slowa:
    status = "✓ jest" if s in wv else "✗ brak"
    print(f"  {status}  '{s}'")

---
## 2. Podobieństwo między słowami

Model zwraca liczbę od -1 do 1 (podobieństwo cosinusowe).

In [ ]:
# Pary słów i ich podobieństwo
pary = [
    ("ojciec",  "matka"),
    ("ojciec",  "syn"),
    ("ojciec",  "rzeka"),
    ("król",    "władca"),
    ("miłość",  "nienawiść"),
    ("miłość",  "czułość"),
    ("bitwa",   "pokój"),
    ("miecz",   "pióro"),
]

print(f"{'Para':35} {'Podobieństwo':>12}  Wizualizacja")
print("-" * 70)
for w1, w2 in pary:
    if w1 in wv and w2 in wv:
        s = wv.similarity(w1, w2)
        bar_len = int((s + 1) / 2 * 25)  # skaluj -1..1 → 0..25
        bar = "█" * bar_len + "░" * (25 - bar_len)
        print(f"  {w1} ↔ {w2:25} {s:>8.4f}  [{bar}]")

---
## 3. Najbliżsi sąsiedzi

In [ ]:
# Dla dowolnego słowa – znajdź 10 najbardziej podobnych
def sasiedzi(slowo, n=10):
    if slowo not in wv:
        print(f"'{slowo}' nie ma w słowniku")
        return
    wyniki = wv.most_similar(slowo, topn=n)
    print(f"\nNajbliższe słowa do '{slowo}':")
    for i, (w, s) in enumerate(wyniki, 1):
        print(f"  {i:2}. {w:20} {s:.4f}")

sasiedzi("bóg")
sasiedzi("serce")
sasiedzi("śmierć")

---
## 4. Analogie – arytmetyka na wektorach

`positive` = słowa, które **dodajemy**  
`negative` = słowa, które **odejmujemy**

In [ ]:
def analogia(positive, negative, n=5):
    brak = [w for w in positive + negative if w not in wv]
    if brak:
        print(f"Brak w słowniku: {brak}")
        return
    wyniki = wv.most_similar(positive=positive, negative=negative, topn=n)
    plus  = " + ".join(positive)
    minus = " - ".join(negative)
    print(f"{plus} - {minus} = ?")
    for w, s in wyniki:
        print(f"  → {w:20} ({s:.4f})")
    print()

# Hierarchia władzy i płeć
analogia(["król",    "kobieta"],  ["mężczyzna"])
analogia(["książę",  "kobieta"],  ["mężczyzna"])

# Czas dnia
analogia(["noc",    "słońce"],   ["księżyc"])

# Relacje rodzinne
analogia(["ojciec", "córka"],    ["syn"])

# Natura
analogia(["morze",  "rzeka"],    ["ocean"])

---
## 5. Znajdź intruza

Które słowo semantycznie **nie pasuje** do reszty?

In [ ]:
def intruz(slowa):
    dostepne = [w for w in slowa if w in wv]
    brak = set(slowa) - set(dostepne)
    if brak:
        print(f"  Pominięto (brak): {brak}")
    if len(dostepne) < 3:
        print("  Za mało słów")
        return
    wynik = wv.doesnt_match(dostepne)
    print(f"{dostepne}  →  intruz: '{wynik}'")

intruz(["rzeka", "morze", "jezioro", "staw",  "miecz"])
intruz(["radość", "szczęście", "miłość", "bitwa", "nadzieja"])
intruz(["koń", "pies", "kot", "wilk", "chleb"])
intruz(["król", "cesarz", "hetman", "sułtan", "chłop"])
intruz(["matka", "ojciec", "córka", "syn", "miecz"])

---
## 6. Podobieństwo zdań (uśrednianie wektorów)

Prosty sposób na reprezentację zdania: **uśrednij wektory słów**.  
Potem można mierzyć podobieństwo między zdaniami.

In [ ]:
import re

def zdanie_na_wektor(tekst):
    """Tokenizuje zdanie i zwraca uśredniony wektor słów obecnych w modelu."""
    tokeny = re.sub(r'[^a-ząćęłńóśźż\s]', ' ', tekst.lower()).split()
    wektory = [wv[t] for t in tokeny if t in wv]
    if not wektory:
        return None
    return np.mean(wektory, axis=0)

def podobienstwo_zdan(z1, z2):
    """Podobieństwo cosinusowe dwóch zdań."""
    v1 = zdanie_na_wektor(z1)
    v2 = zdanie_na_wektor(z2)
    if v1 is None or v2 is None:
        return None
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


zdania = [
    "Król zasiadł na tronie i wydał rozkaz.",
    "Władca objął rządy i przemówił do ludu.",
    "Rycerz dobył miecza i ruszył do bitwy.",
    "Wojownik wziął broń i stanął do walki.",
    "Dziewczyna biegła przez łąkę i śpiewała.",
]

print("Podobieństwo między zdaniami (macierz):\n")
print(f"{'':4}", end="")
for i in range(len(zdania)):
    print(f"  Z{i+1}  ", end="")
print()

for i, z1 in enumerate(zdania):
    print(f"Z{i+1} ", end="")
    for z2 in zdania:
        s = podobienstwo_zdan(z1, z2)
        print(f" {s:.3f}", end="")
    print(f"  ← {z1[:45]}..." if len(z1) > 45 else f"  ← {z1}")

---
## 7. Klasteryzacja słów (K-Means)

Pogrupuj 200 najczęstszych słów automatycznie – bez podawania nazw grup.

In [ ]:
from sklearn.cluster import KMeans

# Weź N najczęstszych słów (pomijając stopwords)
STOPWORDS = {"się", "nie", "to", "że", "i", "w", "z", "na", "do", "jak",
             "go", "jej", "jego", "już", "ale", "bo", "co", "a", "o", "pan",
             "tak", "ten", "ta", "te", "by", "mu", "mi", "też", "po", "czy"}

TOP_N = 300
N_CLUSTERS = 8

slowa_top = [w for w in wv.index_to_key if w not in STOPWORDS][:TOP_N]
wektory = wv[slowa_top]

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
etykiety = kmeans.fit_predict(wektory)

# Pokaż zawartość każdego klastra
for k in range(N_CLUSTERS):
    slowa_k = [slowa_top[i] for i, e in enumerate(etykiety) if e == k]
    print(f"\nKlaster {k+1} ({len(slowa_k)} słów):")
    print("  " + ", ".join(slowa_k[:20]))

---
## 8. Wizualizacja t-SNE klastrów

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from sklearn.manifold import TSNE

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Redukuj do 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
pts = tsne.fit_transform(wektory)

kolory = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(15, 10))
for i, (x, y) in enumerate(pts):
    k = etykiety[i]
    ax.scatter(x, y, color=kolory[k % len(kolory)], s=60, zorder=2, alpha=0.8)
    ax.annotate(slowa_top[i], (x, y), fontsize=7.5, ha='center', va='bottom',
                xytext=(0, 3), textcoords='offset points', color=kolory[k % len(kolory)])

ax.set_title(f"K-Means ({N_CLUSTERS} klastrów) na {TOP_N} najczęstszych słowach – t-SNE",
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 9. Wyszukiwanie semantyczne

Podaj zapytanie w naturalnym języku – znajdź najbardziej "pasujące" słowa z modelu.

In [ ]:
def szukaj(zapytanie, n=10):
    """
    Traktuje zapytanie jako bag-of-words, uśrednia wektory
    i zwraca n słów z modelu o najwyższym podobieństwie.
    """
    tokeny = re.sub(r'[^a-ząćęłńóśźż\s]', ' ', zapytanie.lower()).split()
    wektory_q = [wv[t] for t in tokeny if t in wv]
    if not wektory_q:
        print("Żadne słowo zapytania nie jest w słowniku")
        return
    q_vec = np.mean(wektory_q, axis=0)
    wyniki = wv.similar_by_vector(q_vec, topn=n + len(tokeny))
    # odfiltruj słowa z zapytania
    wyniki = [(w, s) for w, s in wyniki if w not in tokeny][:n]
    print(f"\nZapytanie: '{zapytanie}'")
    for i, (w, s) in enumerate(wyniki, 1):
        print(f"  {i:2}. {w:20} {s:.4f}")

szukaj("walka i chwała rycerza")
szukaj("miłość i tęsknota")
szukaj("przyroda rzeka las")
szukaj("bóg modlitwa wiara")

---
## 10. Interpolacja między słowami

Przejdź płynnie od jednego słowa do drugiego w przestrzeni wektorowej.

In [ ]:
def interpoluj(slowo_a, slowo_b, kroki=6):
    """Wyświetl słowa leżące na 'drodze' od słowa A do słowa B."""
    if slowo_a not in wv or slowo_b not in wv:
        print("Jedno ze słów nie jest w słowniku")
        return
    va = wv[slowo_a]
    vb = wv[slowo_b]
    print(f"\nInterpolacja: {slowo_a} → {slowo_b}")
    for i, t in enumerate(np.linspace(0, 1, kroki)):
        v = (1 - t) * va + t * vb
        najblizsze = wv.similar_by_vector(v, topn=3)
        # pierwsze słowo które nie jest ani A ani B
        dla_kroku = [w for w, _ in najblizsze if w not in (slowo_a, slowo_b)]
        etykieta = slowo_a if i == 0 else (slowo_b if i == kroki - 1 else dla_kroku[0] if dla_kroku else "?")
        prog = int(t * 30)
        print(f"  t={t:.2f}  [{'█'*prog + '░'*(30-prog)}]  {etykieta}")

interpoluj("miłość", "nienawiść")
interpoluj("król",   "chłop")
interpoluj("dzień",  "noc")

---
## 11. Macierz podobieństwa dla grupy słów

In [ ]:
slowa_grupa = ["miłość", "nienawiść", "przyjaźń", "radość", "smutek",
               "gniew", "strach", "nadzieja", "zazdrość", "żal"]

dostepne = [w for w in slowa_grupa if w in wv]
n = len(dostepne)
macierz = np.zeros((n, n))

for i, w1 in enumerate(dostepne):
    for j, w2 in enumerate(dostepne):
        macierz[i][j] = wv.similarity(w1, w2)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(macierz, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(dostepne, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(dostepne, fontsize=10)

for i in range(n):
    for j in range(n):
        val = macierz[i][j]
        kolor = 'black' if 0.2 < val < 0.8 else 'white'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=kolor)

plt.colorbar(im, ax=ax, label='Podobieństwo cosinusowe')
ax.set_title('Macierz podobieństwa – emocje i stany uczuciowe', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()